# SECOM: preparing data for a Tableau dashboard

This notebook builds the table behind my Tableau dashboard on the SECOM semiconductor
dataset from UCI (1,567 samples, about 3 months in 2008).

The loading, cleaning and analysis were done in my separate
[SECOM analysis project](https://github.com/yuning777/secom-process-analysis), so I do not repeat them here.
The input file `secom_clean.csv` comes from that project. In it, I already dropped
116 sensors with the same value in every sample and 28 sensors that were missing more
than half of their values, which leaves 446 sensors.

A dashboard cannot show 446 sensors, so this notebook only does three things:
1. Pick 6 sensors to show (the reasons are in the README).
2. Add a few columns that make Tableau easier to use: `result` (Pass / Fail),
   `is_fail` (0 or 1, so the average is the fail rate), `date`, and `period`
   (before or after Aug 25, where the fail rate drops from 12.9% to 4.1% in my earlier analysis).
3. Save everything as one flat CSV, with one row per sample.

I did not fill in missing sensor values. They stay empty and Tableau treats them as nulls.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

# secom_clean.csv comes from my SECOM analysis project
df_clean = pd.read_csv("../data/secom_clean.csv", parse_dates=["timestamp"])
print(df_clean.shape)   # expected: (1567, 448)

In [ ]:
# Sensors for the dashboard:
# sensor_59, 477 and 205 still differ between pass and fail after removing drift over time;
# sensor_103, 510 and 129 looked linked to failure only because they drift over time.
selected = ["sensor_59", "sensor_477", "sensor_205",
            "sensor_103", "sensor_510", "sensor_129"]

dash = df_clean[["timestamp", "label"] + selected].copy()
dash["result"] = dash["label"].map({-1: "Pass", 1: "Fail"})
dash["is_fail"] = (dash["label"] == 1).astype(int)   # AVG(is_fail) = fail rate in Tableau
dash["date"] = dash["timestamp"].dt.date
dash["period"] = np.where(dash["timestamp"] < pd.Timestamp("2008-08-25"),
                          "Early (before Aug 25)", "Late (Aug 25 on)")

Path("../data/processed").mkdir(parents=True, exist_ok=True)
dash.drop(columns=["label"]).to_csv("../data/processed/secom_dashboard.csv", index=False)
print(dash.shape) 
dash.head()

(1567, 12)


,timestamp,label,sensor_59,sensor_477,sensor_205,sensor_103,sensor_510,sensor_129,result,is_fail,date,period
0,2008-07-19 11:55:00,-1,-1.7264,5.8142,8.64,-0.0042,64.6707,-0.0473,Pass,0,2008-07-19,Early (before Aug 25)
1,2008-07-19 12:32:00,-1,0.8073,5.8042,8.96,-0.0045,141.4365,-0.0946,Pass,0,2008-07-19,Early (before Aug 25)
2,2008-07-19 13:17:00,1,23.8245,9.1221,13.30,-0.0026,240.7767,-0.1892,Fail,1,2008-07-19,Early (before Aug 25)
3,2008-07-19 14:43:00,-1,24.3791,6.4746,9.56,-0.0059,113.5593,0.2838,Pass,0,2008-07-19,Early (before Aug 25)
4,2008-07-19 15:22:00,-1,-12.2945,6.1979,9.09,-0.0045,148.0663,-0.5677,Pass,0,2008-07-19,Early (before Aug 25)
